In [1]:
import matplotlib.pyplot as plt
import matplotlib.dates as mdates
import matplotlib.ticker as mticker
import numpy as np
import time
import netCDF4
from netCDF4 import Dataset
from datetime import datetime, timezone
import cartopy.crs as ccrs
import mpl_toolkits
from mpl_toolkits.basemap import Basemap
import cartopy
import cartopy.crs as ccrs
import cartopy.feature as cfeat
from cartopy.mpl.gridliner import LONGITUDE_FORMATTER, LATITUDE_FORMATTER
from scipy.interpolate import griddata

## Declare Input netCDF-4 File

In [2]:
prefix="ZCYL5_20211110v30001"
nc_file="./data/"+prefix+".nc"

## Open netCDF4 file for reading

In [3]:
try:
    nc_file1 = Dataset(nc_file, 'r' )
except IOError:
    print ('not a valid netCDF file')


## Open with the netCDF file, starting with reading in the Global Attributes.

In [4]:
def readGlobalAttrs( nc_file1 ):

  attr_list = []
  global_attr = []
  for attr_name in nc_file1.ncattrs():
    attr_list.append( attr_name )
    atts = getattr( nc_file1, attr_name )
    global_attr.append( atts )
  return attr_list, global_attr 

## Print the netCDF Global Attributes

In [5]:
print ('Global attributes:')
[attr_list, global_attr] = readGlobalAttrs( nc_file1 )

natts = len(attr_list)
for n in range(0, natts):
     print (attr_list[n]," = ",global_attr[n])

Global attributes:
title  =  FALKOR Meteorological Data
site  =  FALKOR
elev  =  0
ID  =  ZCYL5
IMO  =  007928677
platform  =  unknown at this time
platform_version  =  unknown at this time
facility  =  Schmidt Ocean Institute
data_provider  =  Stian Alesandrini
contact_info  =  Center for Ocean-Atmospheric Prediction Studies, The Florida State University, Tallahassee, FL, 32306-2840, USA
contact_email  =  samos@coaps.fsu.edu
fsu_version  =  300
receipt_order  =  01
start_date_time  =  2021/11/10 -- 00:00 UTC
end_date_time  =  2021/11/10 -- 16:02 UTC
EXPOCODE  =  EXPOCODE undefined for now
Cruise_id  =  Cruise_id undefined for now
Data_modification_date  =  11/30/2021 15:45:00 EST
Metadata_modification_date  =  11/30/2021 15:45:00 EST
metadata_retrieved_from  =  ZCYL5_20211110v10003.nc
files_merged  =  [ZCYL5_20211110v10001.nc, ZCYL5_20211110v10002.nc, ZCYL5_20211110v10003.nc]
merger_version  =  v001


## Read the netCDF Variables and Attribute Metadata

In [6]:
var_list = []
var_attr_list = []
var_data_list = []
def readVars ( nc_file1 ):
      vars =  nc_file1.variables.keys()
      for var_name in vars:
          attr = nc_file1.variables[var_name]
          vardata = nc_file1.variables[var_name][:]
          var_attr_list.append( attr )
          var_data_list.append( vardata)
      return vars, var_attr_list, var_data_list

## Print the netCDF Variable Attributes

In [7]:
print ('Variable attributes:')
[vars, var_attr_list, var_data_list] = readVars( nc_file1 )
nvars = len(vars)
print ('Number of variables = ', nvars)
for i in range(0, nvars):
    vardata = var_data_list[i]
    print ('----------')
    print (var_attr_list[i])

Variable attributes:
Number of variables =  35
----------
<class 'netCDF4.Variable'>
int32 time(time)
    long_name: time
    units: minutes since 1-1-1980 00:00 UTC
    original_units: hhmmss UTC
    data_interval: 60
    observation_type: measured
    actual_range: [22016160 22017122]
    qcindex: 1
    metadata_retrieved_from: ZCYL5_20211110v10003.nc
unlimited dimensions: time
current shape = (963,)
filling on, default _FillValue of -2147483647 used
----------
<class 'netCDF4.Variable'>
float32 lat(time)
    actual_range: [23.847445 24.520512]
    long_name: latitude
    units: degrees (+N)
    original_units: degrees (+N)
    instrument: C & C Technologies CNAV3050
    observation_type: measured
    average_method: average
    average_center: time at end of period
    average_length: 60
    sampling_rate: 1.0
    data_precision: 1e-06
    qcindex: 2
    metadata_retrieved_from: ZCYL5_20211110v10003.nc
unlimited dimensions: time
current shape = (963,)
filling on, default _FillValue 

## Declare New Test netCDF-4 File for Output 

In [8]:
outfile="./data/"+prefix+"_cf_test.nc"

## Define values for history global attribute

In [25]:
now = datetime.now()
cdt_str = now.strftime("%Y-%m-%d"+"T"+"%H:%M:%S")
history_str = "Modified from the original SAMOS fsu_version 300 file, "+prefix+".nc, on "+cdt_str
history_str

'Modified from the original SAMOS fsu_version 300 file, ZCYL5_20211110v30001.nc, on 2026-07-15T18:32:54'

## Execute NCO Commands to update metadata for CF Compliance

In [34]:
!echo $nc_file
!echo $outfile
!echo {history_str}
# Start by appending the standard_name attributes to relevant variables, then units updates, then global attriubtes updates.
!ncatted -a standard_name,time,a,c,time \
-a standard_name,lat,a,c,latitude \
-a standard_name,lon,a,c,longitude \
-a standard_name,PL_HD,a,c,platform_orientation \
-a standard_name,PL_CRS,a,c,platform_course \
-a standard_name,DIR,a,c,wind_from_direction \
-a standard_name,DIR2,a,c,wind_from_direction \
-a standard_name,DIR3,a,c,wind_from_direction \
-a standard_name,PL_WDIR,a,c,wind_from_direction \
-a standard_name,PL_WDIR2,a,c,wind_from_direction \
-a standard_name,PL_WDIR3,a,c,wind_from_direction \
-a standard_name,PL_SPD,a,c,platform_speed_wrt_ground \
-a standard_name,SPD,a,c,wind_speed \
-a standard_name,SPD2,a,c,wind_speed \
-a standard_name,SPD3,a,c,wind_speed \
-a standard_name,PL_WSPD,a,c,wind_speed \
-a standard_name,PL_WSPD2,a,c,wind_speed \
-a standard_name,PL_WSPD3,a,c,wind_speed \
-a standard_name,P,a,c,air_pressure \
-a standard_name,P2,a,c,air_pressure \
-a standard_name,P3,a,c,air_pressure \
-a standard_name,T,a,c,air_temperature \
-a standard_name,T2,a,c,air_temperature \
-a standard_name,T3,a,c,air_temperature \
-a standard_name,RH,a,c,relative_humidity \
-a standard_name,RH2,a,c,relative_humidity \
-a standard_name,RH3,a,c,relative_humidity \
-a standard_name,TS,a,c,sea_water_temperature \
-a standard_name,TS2,a,c,sea_water_temperature \
-a standard_name,SSPS,a,c,sea_water_salinity \
-a standard_name,CNDC,a,c,sea_water_electrical_conductivity \
-a standard_name,date,a,c,time \
-a standard_name,time_of_day,a,c,time \
-a units,lat,m,c,degrees_north \
-a units,lon,m,c,degrees_east \
-a units,PL_HD,m,c,degrees \
-a units,PL_CRS,m,c,degrees \
-a units,DIR,m,c,degrees \
-a units,DIR2,m,c,degrees \
-a units,DIR3,m,c,degrees \
-a units,PL_WDIR,m,c,degrees \
-a units,PL_WDIR2,m,c,degrees \
-a units,PL_WDIR3,m,c,degrees \
-a units,SSPS,m,c,"1e-3" \
-a calendar,time,a,c,"standard" \
-a calendar,date,a,c,"standard" \
-a calendar,time_of_day,a,c,"standard" \
-a Conventions,"global",a,c,"CF-1.9" \
-a institution,"global",a,c,"Marine Data Center, Center for Ocean-Atmospheric Prediction Studies, Florida State University" \
-a history,"global",a,c,"{history_str}" \
-a references,"global",a,c,"Smith, S. R., K. Briggs, M. A. Bourassa, J. Elya, and C. R. Paver, 2018: Shipboard automated meteorological and oceanographic system data archive: 2005-2017. Geosci Data J., 6:354, DOI: https://doi.org/10.1002/gdj3.59" \
-a comment,"global",a,c,"This is a CF-compliant transformation of the original SAMOS file." \
-H -h $nc_file -O -o $outfile

./data/ZCYL5_20211110v30001.nc
./data/ZCYL5_20211110v30001_cf_test.nc
Modified from the original SAMOS fsu_version 300 file, ZCYL5_20211110v30001.nc, on 2026-07-15T18:32:54


## Open New netCDF CF Test File, Starting with Global Attributes

In [35]:
try:
    nc_file2 = Dataset(outfile, 'r' )
except IOError:
    print ('not a valid netCDF file')

def readGlobalAttrs( outfile ):

  attr_list = []
  global_attr = []
  for attr_name in nc_file2.ncattrs():
    attr_list.append( attr_name )
    atts = getattr( nc_file2, attr_name )
    global_attr.append( atts )
  return attr_list, global_attr 

## Print the netCDF CF Test File Global Attributes

In [36]:
print ('Global attributes:')
[attr_list, global_attr] = readGlobalAttrs( nc_file2 )

natts = len(attr_list)
for n in range(0, natts):
     print (attr_list[n]," = ",global_attr[n])

Global attributes:
title  =  FALKOR Meteorological Data
site  =  FALKOR
elev  =  0
ID  =  ZCYL5
IMO  =  007928677
platform  =  unknown at this time
platform_version  =  unknown at this time
facility  =  Schmidt Ocean Institute
data_provider  =  Stian Alesandrini
contact_info  =  Center for Ocean-Atmospheric Prediction Studies, The Florida State University, Tallahassee, FL, 32306-2840, USA
contact_email  =  samos@coaps.fsu.edu
fsu_version  =  300
receipt_order  =  01
start_date_time  =  2021/11/10 -- 00:00 UTC
end_date_time  =  2021/11/10 -- 16:02 UTC
EXPOCODE  =  EXPOCODE undefined for now
Cruise_id  =  Cruise_id undefined for now
Data_modification_date  =  11/30/2021 15:45:00 EST
Metadata_modification_date  =  11/30/2021 15:45:00 EST
metadata_retrieved_from  =  ZCYL5_20211110v10003.nc
files_merged  =  [ZCYL5_20211110v10001.nc, ZCYL5_20211110v10002.nc, ZCYL5_20211110v10003.nc]
merger_version  =  v001
Conventions  =  CF-1.9
institution  =  Marine Data Center, Center for Ocean-Atmospheri

## Read the netCDF CF Test FIle Variables and Attribute Metadata

In [17]:
var_list = []
var_attr_list = []
var_data_list = []
def readVars ( nc_file2 ):
      vars =  nc_file2.variables.keys()
      for var_name in vars:
          attr = nc_file2.variables[var_name]
          vardata = nc_file2.variables[var_name][:]
          var_attr_list.append( attr )
          var_data_list.append( vardata)
      return vars, var_attr_list, var_data_list

## Print the netCDF CF Test File Variable Attributes

In [18]:
print ('netCDF CF Test File Variable attributes:')
[vars, var_attr_list, var_data_list] = readVars( nc_file2 )
nvars = len(vars)
print ('Number of variables = ', nvars)
for i in range(0, nvars):
    vardata = var_data_list[i]
    print ('----------')
    print (var_attr_list[i])

netCDF CF Test File Variable attributes:
Number of variables =  35
----------
<class 'netCDF4.Variable'>
int32 time(time)
    long_name: time
    units: minutes since 1-1-1980 00:00 UTC
    original_units: hhmmss UTC
    data_interval: 60
    observation_type: measured
    actual_range: [22016160 22017122]
    qcindex: 1
    metadata_retrieved_from: ZCYL5_20211110v10003.nc
    standard_name: time
    calendar: standard
unlimited dimensions: time
current shape = (963,)
filling on, default _FillValue of -2147483647 used
----------
<class 'netCDF4.Variable'>
float32 lat(time)
    actual_range: [23.847445 24.520512]
    long_name: latitude
    units: degrees_north
    original_units: degrees (+N)
    instrument: C & C Technologies CNAV3050
    observation_type: measured
    average_method: average
    average_center: time at end of period
    average_length: 60
    sampling_rate: 1.0
    data_precision: 1e-06
    qcindex: 2
    metadata_retrieved_from: ZCYL5_20211110v10003.nc
    standard_